# Experiment 04 — Horizon Analysis

**Goal:** Evaluate how forecast quality degrades with increasing
prediction horizon.

Horizons evaluated: **6h, 12h, 24h, 48h, 72h**

For each horizon, a **separate BiLSTM** model is trained from scratch
with the correct output dimension.  This avoids the methodological error
of reusing a 24 h model for different horizons.

In [ ]:
# ── Cell 1: Environment Setup ────────────────────────────────────
from pathlib import Path
import subprocess, sys

PROJECT_ROOT = Path("/kaggle/working/stlf-entso-2026")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/AlvinHarist/stlf-entso-2026.git",
         str(PROJECT_ROOT)],
        check=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────
import platform, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import tensorflow as tf

from src.utils.seed import set_seed
from src.data.load_data import load_dataset
from src.data.preprocessing import chronological_split, fit_preprocessor, transform_data, inverse_y
from src.data.windowing import create_train_windows, create_evaluation_windows
from src.models.bilstm import build_bilstm
from src.training.trainer import train_model
from src.evaluation.point_metrics import compute_all_metrics, horizon_wise_metrics

print(f"Python: {platform.python_version()} | TF: {tf.__version__}")

In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────
CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

SEED       = config["seed"]
LOOKBACK   = config["windowing"]["lookback"]
TARGET_COL = config["data"]["target_col"]
UNITS      = config["model"]["units"]
DROPOUT    = config["model"]["dropout"]
LR         = config["model"]["learning_rate"]
EPOCHS     = config["training"]["epochs"]
BATCH_SIZE = config["training"]["batch_size"]
PATIENCE   = config["training"]["patience"]

HORIZONS = [6, 12, 24, 48, 72]

# Data path
KAGGLE_DATA_DIR = Path("/kaggle/input/stlf-entso-2026")
DATA_PATH = None
if KAGGLE_DATA_DIR.exists():
    for p in KAGGLE_DATA_DIR.rglob("*.csv"):
        if "combined_AT" in p.name:
            DATA_PATH = p
            break
if DATA_PATH is None:
    for fb in [PROJECT_ROOT / config["data"]["path"], PROJECT_ROOT / "df_combined_AT.csv"]:
        if fb.exists():
            DATA_PATH = fb
            break
if DATA_PATH is None:
    raise FileNotFoundError("Cannot locate df_combined_AT.csv")

RESULTS_DIR = PROJECT_ROOT / "results" / "deterministic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dataset: {DATA_PATH}")
print(f"Horizons: {HORIZONS}")

In [ ]:
# ── Cell 4: Load & Preprocess (once) ─────────────────────────────
df = load_dataset(DATA_PATH, timestamp_col=config["data"]["timestamp_col"])
train_df, val_df, test_df = chronological_split(
    df, config["split"]["train_ratio"], config["split"]["val_ratio"]
)

# Univariate
feature_cols = [TARGET_COL]
preprocessor = fit_preprocessor(
    train_df, TARGET_COL, feature_cols, use_yeojohnson=False
)

X_tr, y_tr = transform_data(train_df, preprocessor)
X_va, y_va = transform_data(val_df, preprocessor)
X_te, y_te = transform_data(test_df, preprocessor)

In [ ]:
# ── Cell 5: Train & Evaluate Each Horizon ────────────────────────
horizon_results = {}

for H in HORIZONS:
    print(f"\n{'='*60}")
    print(f"  HORIZON = {H}h")
    print(f"{'='*60}")
    
    set_seed(SEED)
    
    # Windows
    Xw_tr, yw_tr = create_train_windows(X_tr, y_tr, LOOKBACK, H)
    Xw_va, yw_va = create_evaluation_windows(X_va, y_va, X_tr, y_tr, LOOKBACK, H)
    Xw_te, yw_te = create_evaluation_windows(X_te, y_te, X_va, y_va, LOOKBACK, H)
    
    print(f"  Windows — Train: {Xw_tr.shape}, Val: {Xw_va.shape}, Test: {Xw_te.shape}")
    
    # Build model with correct output dimension
    mdl = build_bilstm(LOOKBACK, Xw_tr.shape[2], H, UNITS, DROPOUT, LR)
    
    # Train
    tr_res = train_model(mdl, Xw_tr, yw_tr, Xw_va, yw_va, EPOCHS, BATCH_SIZE, PATIENCE, verbose=0)
    
    # Predict & evaluate
    pred_scaled = mdl.predict(Xw_te)
    pred_mw = inverse_y(pred_scaled, preprocessor)
    actual_mw = inverse_y(yw_te, preprocessor)
    
    mets = compute_all_metrics(actual_mw, pred_mw)
    hw = horizon_wise_metrics(actual_mw, pred_mw)
    
    horizon_results[H] = {
        "metrics": mets,
        "horizon_mape": hw["mape"],
        "horizon_mae": hw["mae"],
        "horizon_rmse": hw["rmse"],
        "best_epoch": tr_res["best_epoch"],
    }
    
    print(f"  MAE={mets['MAE']:.2f}  RMSE={mets['RMSE']:.2f}  MAPE={mets['MAPE']:.2f}%  Best epoch={tr_res['best_epoch']}")

In [ ]:
# ── Cell 6: Summary Table ────────────────────────────────────────
print("\n" + "=" * 60)
print("HORIZON ANALYSIS SUMMARY")
print("=" * 60)
print(f"{'Horizon':>10s} {'MAE':>10s} {'RMSE':>10s} {'MAPE':>10s} {'sMAPE':>10s}")
print("-" * 52)

rows = []
for H in HORIZONS:
    m = horizon_results[H]["metrics"]
    print(f"{H:>10d}h {m['MAE']:>10.2f} {m['RMSE']:>10.2f} {m['MAPE']:>10.2f} {m['sMAPE']:>10.2f}")
    rows.append({"horizon": H, **m})

# Save as CSV
pd.DataFrame(rows).to_csv(RESULTS_DIR / "horizon_results.csv", index=False)
print(f"\nSaved to {RESULTS_DIR / 'horizon_results.csv'}")

In [ ]:
# ── Cell 7: Visualisation ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Aggregate metrics vs horizon
for idx, metric_name in enumerate(["MAE", "RMSE", "MAPE"]):
    vals = [horizon_results[H]["metrics"][metric_name] for H in HORIZONS]
    axes[idx].plot(HORIZONS, vals, "o-", linewidth=2)
    axes[idx].set_title(f"{metric_name} vs Horizon")
    axes[idx].set_xlabel("Horizon (h)")
    axes[idx].set_ylabel(metric_name)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
fig.savefig(RESULTS_DIR / "horizon_analysis.png", dpi=150)
plt.show()

# Horizon-wise MAPE for each model
fig2, ax2 = plt.subplots(figsize=(12, 5))
for H in HORIZONS:
    hw_mape = horizon_results[H]["horizon_mape"]
    ax2.plot(range(1, H + 1), hw_mape, "o-", label=f"{H}h model", markersize=3)
ax2.set_title("Step-wise MAPE by Horizon Model")
ax2.set_xlabel("Forecast Step")
ax2.set_ylabel("MAPE (%)")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
fig2.savefig(RESULTS_DIR / "horizon_stepwise_mape.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 8: Save Full Results ────────────────────────────────────
full_rec = {
    "experiment": "04_horizon_analysis",
    "model": "BiLSTM",
    "features": "univariate",
    "lookback": LOOKBACK,
    "seed": SEED,
    "horizons": {}
}
for H in HORIZONS:
    full_rec["horizons"][str(H)] = {
        "metrics": horizon_results[H]["metrics"],
        "best_epoch": horizon_results[H]["best_epoch"],
    }

out_path = RESULTS_DIR / f"horizon_analysis_seed{SEED}.json"
with open(out_path, "w") as f:
    json.dump(full_rec, f, indent=2)
print(f"Saved: {out_path}")